In [ ]:
# セル 1: 必要なライブラリのインポート (変更なし)
import glob
import os

import cupy as cp  # Cupyを使用する場合
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

# from tensorflow.keras.applications import VGG16 # 今回は直接使わない
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (  # Flatten を追加
    Conv2D,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    Input,
    MaxPooling2D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import img_to_array, load_img


In [ ]:
# 再現性のための乱数シード設定
SEED = 42

# Python標準のrandomモジュールのシード設定
import random

random.seed(SEED)

# NumPyのシード設定
np.random.seed(SEED)

# TensorFlowのシード設定
tf.random.set_seed(SEED)

# TensorFlowの決定的動作を有効化（可能な場合）
# 注: 一部の演算では完全な決定性が保証されない場合があります
try:
    tf.config.experimental.enable_op_determinism()
except AttributeError:
    # TensorFlow 2.9未満では利用できない
    pass

# 環境変数でPythonハッシュのシードを固定
os.environ["PYTHONHASHSEED"] = str(SEED)

# CuPyのシード設定（GPU使用時）
try:
    cp.random.seed(SEED)
except:
    pass

# セル 2: 設定値
# IMG_BASE_PATH = "20250425_data/Material_d_Gray/"  # 旧パス（ノートブックディレクトリから見ると不正確）
# CSV_BASE_PATH = "20250425_data/TorqData/"  # 旧パス（ノートブックディレクトリから見ると不正確）
IMG_BASE_PATH = "../20250425_data/Material_d_Gray/"  # ノートブックは notebook/ 配下にあるため親ディレクトリを指定
CSV_BASE_PATH = "../20250425_data/TorqData/"  # ノートブックは notebook/ 配下にあるため親ディレクトリを指定
IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 1  # 元画像のチャネル数 (グレースケール)
# MODEL_INPUT_CHANNELS は不要になるか、1 に設定 (モデル定義に合わせる)
BATCH_SIZE = 128  # 小さくした方が良いかもしれない (32 or 64)
EPOCHS = 100
LEARNING_RATE = (
    1e-4  # スクラッチからの学習なので、少し大きめでも良いかもしれない (1e-3)
)
TEST_SPLIT_RATIO = 0.1
VALID_SPLIT_RATIO = 0.1

# CUDAデバイスの設定 ... (変更なし)
print("利用可能なGPUデバイス (Cupy):")
try:
    for i in range(cp.cuda.runtime.getDeviceCount()):
        print(f"GPU {i}: {cp.cuda.runtime.getDeviceProperties(i)['name']}")
    if cp.cuda.runtime.getDeviceCount() > 0:
        cp.cuda.Device(0).use()
        print(f"\n選択されたCupy GPU: {cp.cuda.runtime.getDeviceProperties(0)['name']}")
    else:
        print("\nCupyが利用できるGPUが見つかりませんでした。")
except cp.cuda.runtime.CUDARuntimeError as e:
    print(f"Cupy CUDAエラー: {e}")

print("\n利用可能なGPUデバイス (TensorFlow):")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"{len(gpus)} 個のGPUがTensorFlowで利用可能です。")
    except RuntimeError as e:
        print(e)
else:
    print("TensorFlowで利用可能なGPUが見つかりませんでした。CPUを使用します。")


In [ ]:
# セル 3: CSVファイルから目的変数を抽出するヘルパー関数 (変更なし)
def get_target_value(csv_filepath):
    try:
        df = pd.read_csv(csv_filepath)
        target_df = df.head(30)
        if len(target_df) < 30:
            # print(f"警告: {csv_filepath} の行数が30未満です ({len(target_df)}行)。利用可能な行で計算します。")
            if len(target_df) == 0:
                # print(f"エラー: {csv_filepath} に有効なデータがありません。")
                return np.nan
        target_mean = target_df["T"].mean()
        return target_mean
    except Exception:
        # print(f"エラー: {csv_filepath} の処理中にエラーが発生しました: {e}")
        return np.nan


In [ ]:
# セル 4: データ読み込みと準備 (画像パスと目的変数)
image_files = sorted(glob.glob(os.path.join(IMG_BASE_PATH, "SingleI_*.bmp")))
all_image_paths = []
all_targets = []

for img_path in image_files:
    basename = os.path.basename(img_path)
    try:
        parts_str = basename.replace("SingleI_", "").replace(".bmp", "")
        parts = parts_str.split("_")
        if len(parts) == 2:
            csv_filename = f"Data_{parts[0]}_{parts[1]}.csv"
            csv_filepath = os.path.join(CSV_BASE_PATH, csv_filename)
            if os.path.exists(csv_filepath):
                target = get_target_value(csv_filepath)
                if not np.isnan(target):
                    all_image_paths.append(img_path)
                    all_targets.append(target)
                else:
                    print(
                        f"スキップ: {img_path} に対応する {csv_filepath} の目的変数が無効です。"
                    )
            else:
                print(
                    f"警告: {img_path} に対応するCSVファイルが見つかりません: {csv_filepath}"
                )
        else:
            print(f"警告: 画像ファイル名の形式が正しくありません: {img_path}")
    except Exception as e:
        print(f"エラー: 画像ファイル名 {img_path} の解析中にエラー: {e}")

all_image_paths = np.array(all_image_paths)
all_targets = np.array(all_targets)

print(f"\n{len(all_image_paths)} 個の画像と対応する目的変数を読み込みました。")
if len(all_image_paths) == 0:
    raise ValueError(
        "画像と目的変数のペアが読み込めませんでした。パスやファイル名、CSVの内容を確認してください。"
    )
print(f"目的変数の例: {all_targets[:5]}")


In [ ]:
# セル 5: データ分割 (変更なし)
train_val_paths, test_paths, train_val_targets, test_targets = train_test_split(
    all_image_paths,
    all_targets,
    test_size=TEST_SPLIT_RATIO,
    random_state=42,
    shuffle=True,
)
if len(train_val_paths) == 0:
    raise ValueError("テストデータ分割後、訓練・検証データが0になりました。")
validation_split_from_train_val = VALID_SPLIT_RATIO / (1.0 - TEST_SPLIT_RATIO)
if not (0 <= validation_split_from_train_val < 1):
    raise ValueError(
        f"検証データの分割比率が無効です: {validation_split_from_train_val}."
    )
train_paths, val_paths, train_targets, val_targets = train_test_split(
    train_val_paths,
    train_val_targets,
    test_size=validation_split_from_train_val,
    random_state=42,
    shuffle=True,
)
print(f"訓練データ数: {len(train_paths)}")
print(f"検証データ数: {len(val_paths)}")
print(f"テストデータ数: {len(test_paths)}")
if len(train_paths) == 0 or len(val_paths) == 0 or len(test_paths) == 0:
    print("警告: 訓練、検証、またはテストデータのいずれかが0件です。")


In [ ]:
# セル 6: 1チャネル用の画像読み込みと前処理関数
# VGG16のpreprocess_inputはRGBをBGRに変換しゼロセンタリングしますが、
# 1チャネルの場合は、単純なスケーリング（例: /255.0）や、
# グレースケール画像の統計に基づく正規化が考えられます。
# ここでは、一般的な /255.0 スケーリングを使用します。
def load_and_preprocess_image_1channel(image_path_tensor):
    image_path_str = image_path_tensor.numpy().decode("utf-8")

    img = load_img(
        image_path_str,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        color_mode="grayscale",  # CHANNELS == 1 を前提
    )
    img_array = img_to_array(img)  # (height, width, 1)

    # 形状チェック (万が一2チャネルなどで読み込まれた場合のエラーハンドリング)
    if img_array.shape[-1] != 1:
        print(
            f"警告: {image_path_str} は1チャネルのはずが {img_array.shape} で読み込まれました。最初のチャネルを使用します。"
        )
        img_array = img_array[..., 0:1]

    # ピクセル値を0-1の範囲にスケーリング
    img_array_scaled = img_array / 255.0
    return tf.cast(img_array_scaled, tf.float32)


In [ ]:
# セル 7: tf.data.Datasetの作成 (1チャネル用に変更)
def create_dataset_1channel(paths, targets, batch_size, shuffle=True):
    if len(paths) == 0:
        print("警告: create_dataset に渡されたpathsの長さが0です。")
        return tf.data.Dataset.from_tensor_slices(
            (tf.constant([], dtype=tf.string), tf.constant([], dtype=tf.float32))
        ).batch(batch_size)

    path_ds = tf.data.Dataset.from_tensor_slices(paths)

    image_ds = path_ds.map(
        lambda x: tf.py_function(
            load_and_preprocess_image_1channel, [x], tf.float32
        ),  # 変更
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    def set_shape(img):
        img.set_shape((IMG_HEIGHT, IMG_WIDTH, CHANNELS))  # CHANNELS は 1
        return img

    image_ds = image_ds.map(set_shape, num_parallel_calls=tf.data.AUTOTUNE)

    label_ds = tf.data.Dataset.from_tensor_slices(tf.cast(targets, tf.float32))
    ds = tf.data.Dataset.zip((image_ds, label_ds))

    if shuffle:
        # 再現性のためにseedを指定
        ds = ds.shuffle(
            buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=False
        )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds


if len(train_paths) > 0:
    train_dataset = create_dataset_1channel(train_paths, train_targets, BATCH_SIZE)
else:
    print("訓練データがありません。")
    train_dataset = None

if len(val_paths) > 0:
    val_dataset = create_dataset_1channel(
        val_paths, val_targets, BATCH_SIZE, shuffle=False
    )
else:
    print("検証データがありません。")
    val_dataset = None

if len(test_paths) > 0:
    test_dataset = create_dataset_1channel(
        test_paths, test_targets, BATCH_SIZE, shuffle=False
    )
else:
    print("テストデータがありません。")
    test_dataset = None

print(f"Train dataset spec: {train_dataset.element_spec if train_dataset else 'None'}")


In [ ]:
# セル 8: モデルの定義 (VGG16風の1チャネル入力カスタムモデル)
def build_vgg_like_1channel_regressor(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),  # CHANNELS は 1
    learning_rate_param=LEARNING_RATE,
):
    input_layer = Input(shape=input_shape, name="image_input")

    # Block 1
    x = Conv2D(64, (3, 3), activation="relu", padding="same", name="block1_conv1")(
        input_layer
    )
    x = Conv2D(64, (3, 3), activation="relu", padding="same", name="block1_conv2")(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name="block1_pool")(x)

    # Block 2
    x = Conv2D(128, (3, 3), activation="relu", padding="same", name="block2_conv1")(x)
    x = Conv2D(128, (3, 3), activation="relu", padding="same", name="block2_conv2")(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name="block2_pool")(x)

    # Block 3
    x = Conv2D(256, (3, 3), activation="relu", padding="same", name="block3_conv1")(x)
    x = Conv2D(256, (3, 3), activation="relu", padding="same", name="block3_conv2")(x)
    x = Conv2D(256, (3, 3), activation="relu", padding="same", name="block3_conv3")(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name="block3_pool")(x)

    # Block 4
    x = Conv2D(512, (3, 3), activation="relu", padding="same", name="block4_conv1")(x)
    x = Conv2D(512, (3, 3), activation="relu", padding="same", name="block4_conv2")(x)
    x = Conv2D(512, (3, 3), activation="relu", padding="same", name="block4_conv3")(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name="block4_pool")(x)

    # Block 5
    x = Conv2D(512, (3, 3), activation="relu", padding="same", name="block5_conv1")(x)
    x = Conv2D(512, (3, 3), activation="relu", padding="same", name="block5_conv2")(x)
    x = Conv2D(512, (3, 3), activation="relu", padding="same", name="block5_conv3")(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name="block5_pool")(x)

    # 回帰のためのカスタム層 (GlobalAveragePooling2D を使用)
    x = GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = Dense(512, activation="relu", name="fc1")(
        x
    )  # VGG16のFC層は通常もっと大きいが、タスクに応じて調整
    x = Dropout(0.5, name="dropout_fc1")(x)
    output_tensor = Dense(1, name="torque_output")(x)  # 出力1ニューロン、活性化なし

    model = Model(
        inputs=input_layer, outputs=output_tensor, name="vgg_like_1channel_regressor"
    )

    optimizer = Adam(learning_rate=learning_rate_param)
    model.compile(
        optimizer=optimizer,
        loss="mean_squared_error",
        metrics=["mean_absolute_error", tf.keras.metrics.RootMeanSquaredError()],
    )
    return model


model = build_vgg_like_1channel_regressor()
model.summary()


In [ ]:
# セル 9: 訓練時のコールバック設定 (変更なし)
early_stopping = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
)
model_checkpoint = ModelCheckpoint(
    "best_vgg_like_1channel_regressor.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=5, min_lr=1e-6, verbose=1
)
callbacks_list = [early_stopping, model_checkpoint, reduce_lr]


In [ ]:
# # セル 10: モデルの訓練 (初期段階: VGG16ベースは凍結)
# print("--- 初期訓練開始 (VGG16ベースは凍結) ---")
# history = model.fit(
#     train_dataset, epochs=EPOCHS, validation_data=val_dataset, callbacks=callbacks_list
# )


In [ ]:
# Cell 11: Plotting Training History
def plot_history(history, title_prefix=""):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="Training Loss (MSE)")
    plt.plot(history.history["val_loss"], label="Validation Loss (MSE)")
    plt.title(f"{title_prefix}Model Loss")
    plt.ylabel("Loss (MSE)")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["mean_absolute_error"], label="Training MAE")
    plt.plot(history.history["val_mean_absolute_error"], label="Validation MAE")
    plt.title(f"{title_prefix}Model Mean Absolute Error (MAE)")
    plt.ylabel("MAE")
    plt.xlabel("Epoch")
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# セル 12: 最良モデルのロード (EarlyStoppingでrestore_best_weights=Trueなら不要な場合もあるが念のため)
print("最良の重みをロードします...")
model.load_weights(
    "best_vgg_like_1channel_regressor.keras"
)  # ModelCheckpointで保存された最良のモデル


In [ ]:
# セル 15: テストセットでのモデル評価
from sklearn.metrics import r2_score  # R2スコア計算のためにインポート

print("\n--- テストセットでの評価 ---")
# (モデルのロード部分はそのまま)
# ...

test_loss, test_mae, test_rmse_metric_value = model.evaluate(
    test_dataset, verbose=1
)  # evaluateの返り値のRMSEに注意
print(f"テストデータの損失 (MSE): {test_loss:.4f}")
print(f"テストデータの平均絶対誤差 (MAE): {test_mae:.4f}")
print(
    f"テストデータのRMSE (Keras Metric): {test_rmse_metric_value:.4f}"
)  # KerasのメトリックとしてのRMSE

# 全テストデータに対する予測を一度に行う
all_test_predictions = []
all_test_true_values = []
for images, labels in test_dataset:  # バッチ処理されているためループで集める
    all_test_predictions.extend(model.predict_on_batch(images).flatten())
    all_test_true_values.extend(labels.numpy().flatten())

all_test_predictions = np.array(all_test_predictions)
all_test_true_values = np.array(all_test_true_values)

# R2スコアの計算
r2 = r2_score(all_test_true_values, all_test_predictions)
print(f"テストデータのR2スコア: {r2:.4f}")

# RMSEの再計算 (numpyベースで、Kerasメトリックと一致するか確認)
manual_rmse = np.sqrt(np.mean((all_test_predictions - all_test_true_values) ** 2))
print(f"テストデータのRMSE (numpy計算): {manual_rmse:.4f}")


In [ ]:
# セル 16: 予測の実行と結果の可視化 (任意)
# (all_test_true_values と all_test_predictions はセル15で計算済みとする)

plt.figure(figsize=(10, 8))  # 少し縦長に
plt.scatter(
    all_test_true_values, all_test_predictions, alpha=0.6, label="Predicted values"
)  # 翻訳
min_val = min(np.min(all_test_true_values), np.min(all_test_predictions))
max_val = max(np.max(all_test_true_values), np.max(all_test_predictions))
plt.plot(
    [min_val, max_val], [min_val, max_val], "r--", lw=2, label="Ideal prediction"
)  # 翻訳

# グラフにR2スコアとRMSEを表示
# (セル15で計算した r2 と manual_rmse を使用)
# R2とRMSEは国際的に通じるため、そのまま使用します
plt.text(
    0.05,
    0.95,
    f"R2: {r2:.3f}\nRMSE: {manual_rmse:.3f}",
    transform=plt.gca().transAxes,  # 軸相対座標
    fontsize=14,
    verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.5", fc="yellow", alpha=0.5),
)


plt.xlabel("Actual Torque Value (FEM [Nm])")  # 翻訳 (元論文の表記に合わせる)
plt.ylabel("Predicted Torque Value (CNN [Nm])")  # 翻訳 (元論文の表記に合わせる)
plt.title("Actual Torque Value vs. Predicted Torque Value on Test Data")  # 翻訳
plt.legend(loc="lower right")
plt.grid(True)
# 軸の範囲を0から開始し、最大値はデータの最大値より少し大きめに設定
plt.xlim(left=0, right=max_val * 1.05)
plt.ylim(bottom=0, top=max_val * 1.05)
plt.gca().set_aspect("equal", adjustable="box")  # アスペクト比を1:1に
plt.show()


## モデルの精度のみ実行したい．
cellの１から7までを実行し以下を実行してください．

In [ ]:
# 訓練済みモデルのファイルパス
MODEL_PATH = "../best_vgg_like_1channel_regressor.keras"  # ★★★ 実際のファイル名に合わせてください ★★★

print(f"\n訓練済みモデル {MODEL_PATH} をロードします...")
try:
    # カスタムモデルをロードする場合、カスタムオブジェクトを渡す必要がないことが多い
    # (モデルアーキテクチャがKerasの標準レイヤーで構成されていれば)
    # もしカスタムオブジェクト（カスタムレイヤーやカスタム損失関数など）があれば、
    # load_model の custom_objects引数で指定する必要があります。
    # 今回の vgg_like_1channel_regressor は標準レイヤーのみなので不要なはず。
    model = tf.keras.models.load_model(MODEL_PATH)
    print("モデルのロードが完了しました。")
    model.summary()  # ロードされたモデルの概要を表示
except Exception as e:
    print(f"エラー: モデルのロードに失敗しました: {e}")
    print(
        "テスト評価を実行する前に、モデルが正しくロードされていることを確認してください。"
    )
    # エラーが発生した場合、以降の処理が失敗する可能性があるため、ここで終了するか、
    # ダミーの model オブジェクトがある場合はそのまま進む（ただし評価結果は無意味になる）
    # exit() # 必要に応じてプログラムを終了

# モデルがロードされた後、必要であればコンパイルします。
# .keras 形式で保存した場合、通常オプティマイザの状態も保存されるため、
# 再コンパイルは必須ではありませんが、学習率などを変更したい場合は行います。
# ここでは、評価のみなので、学習時のコンパイル設定が引き継がれることを期待します。
# もし load_model で警告が出る場合や、オプティマイザの状態が復元されない場合は、
# 以下のように再コンパイルが必要になることがあります。
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), # または評価時の学習率
#               loss='mean_squared_error',
#               metrics=['mean_absolute_error', tf.keras.metrics.RootMeanSquaredError()])


# セル 15: テストセットでのモデル評価
print("\n--- テストセットでの評価 ---")

if (
    "model" in locals()
    and model is not None
    and "test_dataset" in locals()
    and test_dataset is not None
):
    try:
        test_loss, test_mae, test_rmse_metric_value = model.evaluate(
            test_dataset, verbose=1
        )
        print(f"テストデータの損失 (MSE): {test_loss:.4f}")
        print(f"テストデータの平均絶対誤差 (MAE): {test_mae:.4f}")
        print(f"テストデータのRMSE (Keras Metric): {test_rmse_metric_value:.4f}")

        # 全テストデータに対する予測を一度に行う
        all_test_predictions = []
        all_test_true_values = []

        print("テストデータから真の値と予測値を取得しています...")
        for images, labels in test_dataset:  # バッチ処理されているためループで集める
            predictions_batch = model.predict_on_batch(images)
            all_test_predictions.extend(predictions_batch.flatten())
            all_test_true_values.extend(labels.numpy().flatten())

        all_test_predictions = np.array(all_test_predictions)
        all_test_true_values = np.array(all_test_true_values)

        # R2スコアの計算
        if (
            len(all_test_true_values) > 0 and len(all_test_predictions) > 0
        ):  # データがあるか確認
            r2 = r2_score(all_test_true_values, all_test_predictions)
            print(f"テストデータのR2スコア: {r2:.4f}")

            # RMSEの再計算 (numpyベースで、Kerasメトリックと一致するか確認)
            manual_rmse = np.sqrt(
                np.mean((all_test_predictions - all_test_true_values) ** 2)
            )
            print(f"テストデータのRMSE (numpy計算): {manual_rmse:.4f}")

            # (オプション) 予測結果の散布図プロット
            import matplotlib.pyplot as plt

            plt.figure(figsize=(8, 8))
            plt.scatter(
                all_test_true_values,
                all_test_predictions,
                alpha=0.5,
                label="Predicted values",
            )
            min_val = min(all_test_true_values.min(), all_test_predictions.min())
            max_val = max(all_test_true_values.max(), all_test_predictions.max())
            plt.plot(
                [min_val, max_val],
                [min_val, max_val],
                "r--",
                lw=2,
                label="Ideal prediction",
            )
            plt.xlabel("Actual Torque Value (True)")
            plt.ylabel("Predicted Torque Value (Model)")
            plt.title("Actual vs. Predicted Torque on Test Data")
            plt.legend()
            plt.grid(True)
            plt.text(
                0.05,
                0.95,
                f"R2: {r2:.3f}\nRMSE: {manual_rmse:.3f}",
                transform=plt.gca().transAxes,
                fontsize=12,
                verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.5", fc="yellow", alpha=0.5),
            )
            plt.show()

        else:
            print(
                "テストデータまたは予測結果が空のため、R2スコアとRMSE(numpy)の計算をスキップしました。"
            )

    except Exception as e:
        print(f"テスト評価中にエラーが発生しました: {e}")
else:
    print(
        "モデルまたはテストデータセットが正しく準備されていません。評価をスキップします。"
    )


## 訓練データセットから画像データを取得し，予測を行う

In [ ]:
# (必要なインポートや train_dataset が定義されている前提)
import time  # 時間計測用 (任意)

import numpy as np
import tensorflow as tf

print("\n--- Collecting processed training images into variable X ---")

# 訓練データセットから前処理済みの画像データを収集するためのリスト
all_train_images_list = []
all_train_labels_list = []
print("Gathering processed images from train_dataset...")
start_time = time.time()

# train_dataset をイテレートして画像データを抽出
# train_dataset が定義されていない場合は、セル7などを再実行して作成してください。
# (train_dataset がシャッフルされている場合、元の train_paths との順序は保証されません)
try:
    num_batches = tf.data.experimental.cardinality(train_dataset).numpy()
    if (
        num_batches == tf.data.experimental.UNKNOWN_CARDINALITY
        or num_batches == tf.data.experimental.INFINITE_CARDINALITY
    ):
        print("Warning: Could not determine the exact number of batches.")
        num_batches = -1
except:  # tf.data.experimental.cardinality が使えない/失敗する場合
    print(
        "Warning: Could not determine the number of batches using tf.data.experimental.cardinality."
    )
    num_batches = -1

processed_samples = 0
batch_counter = 0
for (
    images_batch,
    images_labels,
) in train_dataset:  # ラベル情報は今回は使用しないので _ で受け取る
    # バッチ内の画像データをNumPy配列に変換してリストに追加
    all_train_images_list.append(images_batch.numpy())
    all_train_labels_list.append(images_labels.numpy())
    processed_samples += images_batch.shape[0]
    batch_counter += 1
    if num_batches > 0:
        print(
            f"Processing batch {batch_counter}/{num_batches}, Samples processed: {processed_samples}",
            end="\r",
        )
    else:
        print(
            f"Processing batch {batch_counter}, Samples processed: {processed_samples}",
            end="\r",
        )


print("\nConcatenating collected image batches...")
# リストに格納された複数のバッチ（NumPy配列）を一つの大きなNumPy配列に結合
if all_train_images_list:
    X_train_images = np.concatenate(all_train_images_list, axis=0)
    # ユーザーが指定した変数名 'X' に代入
    X = X_train_images
    end_time = time.time()
    print(f"\nFinished collecting images in {end_time - start_time:.2f} seconds.")
    print(
        f"Processed training images are now stored in variable 'X' with shape: {X.shape}"
    )
    # メモリ使用量の概算を表示 (GiB単位)
    print(f"Approximate memory usage of X: {X.nbytes / (1024**3):.2f} GiB")

    # 必要であれば、元のリストを削除してメモリを解放
    del all_train_images_list

else:
    print("\nNo images were collected from train_dataset. Variable 'X' is not created.")
    X = None  # XをNoneに設定

# これで変数 X に前処理済みの訓練画像データ (NumPy配列) が格納されました。
# この後の処理で、この変数 X を使用できますが、メモリ使用量に注意してください。


# model.predict() を使って予測を実行
try:
    # batch_size を指定すると、メモリ使用量を抑えられる
    # ご自身の環境のメモリに合わせて調整してください (例: 64, 32, 16)
    # 指定しない場合は、TensorFlowが自動で決定しようとします
    predict_batch_size = 64  # バッチサイズを調整
    print(f"Starting prediction with batch_size={predict_batch_size}...")
    Y_predictions_raw = model.predict(X, batch_size=predict_batch_size, verbose=1)

    # 予測結果は通常 (サンプル数, 1) の形状なので、(サンプル数,) に平坦化する
    Y = Y_predictions_raw.flatten()

    end_time = time.time()
    print(f"\nFinished prediction in {end_time - start_time:.2f} seconds.")
    print(f"Predictions are now stored in variable 'Y' with shape: {Y.shape}")

    # 予測結果の最初のいくつかを表示 (確認用)
    print(f"First 5 predictions in Y: {Y[:5]}")

except Exception as e:
    print(f"\nAn error occurred during prediction: {e}")
    print("Consider reducing the 'predict_batch_size' if this is a memory issue.")
    Y = None  # エラーが発生した場合は Y を None に設定

# これで変数 Y に、入力 X (訓練画像データ) に対する予測値が格納されました。

X = X.reshape(X.shape[0], X.shape[1] * X.shape[2] * X.shape[3])
print(X.shape, Y.shape, all_train_labels_list)


In [ ]:
Y_true_train_data = np.concatenate(all_train_labels_list, axis=0)
print(Y_true_train_data.shape)


## X, Y, 正解値をファイルに書き込む

In [ ]:
import os
import time

import numpy as np

# --- Save X, Y (predictions), and Y_true (actual labels) to a compressed .npz file ---

# 保存するファイル名を定義 (拡張子は .npz)
filename = "gray_training_data_preprocessed_predicted_and_true_labels.npz"  # ファイル名を少し変更して内容を反映
save_directory = "../20250425_data"  # 保存先のディレクトリ
os.makedirs(save_directory, exist_ok=True)  # 保存先ディレクトリがなければ作成
full_filepath = os.path.join(save_directory, filename)

print(f"\n--- Saving data to {full_filepath} ---")

# ★★★ 元の訓練データの正解ラベルを格納している変数名を指定 ★★★
# 例: Y_true_train, train_targets, all_targets_train など、ご自身のコードに合わせてください。
# ここでは 'Y_true_train_data' という名前の変数に正解ラベルが入っていると仮定します。
# この変数は、X や Y と同じサンプル数・同じ順序である必要があります。


if "Y_true_train_data" not in locals() or Y_true_train_data is None:
    print(
        "Warning: 'X' not found. Cannot create dummy Y_true_train_data of appropriate size."
    )
    Y_true_train_data = None


# 変数 X (画像データ), Y (予測値), Y_true_train_data (正解ラベル) が存在するか確認
if (
    "X" in locals()
    and "Y" in locals()
    and "Y_true_train_data" in locals()
    and X is not None
    and Y is not None
    and Y_true_train_data is not None
):
    # Y と Y_true_train_data の形状が一致するか確認 (Yは予測値なのでXのサンプル数と同じはず)
    if X.shape[0] != Y.shape[0] or X.shape[0] != Y_true_train_data.shape[0]:
        print(
            "Error: Mismatch in the number of samples between X, Y, and Y_true_train_data."
        )
        print(
            f"X shape[0]: {X.shape[0]}, Y shape[0]: {Y.shape[0]}, Y_true_train_data shape[0]: {Y_true_train_data.shape[0]}"
        )
    else:
        start_time = time.time()
        try:
            # np.savez_compressed を使用して圧縮形式で保存
            # キーワード引数で各配列に名前を付ける
            np.savez_compressed(
                full_filepath,
                x_data=X,  # 前処理済み画像データ
                y_predicted_data=Y,  # モデルによる予測値
                y_true_actual_data=Y_true_train_data,  # ★★★ 元の訓練データの正解ラベル ★★★
            )

            end_time = time.time()
            print(f"Successfully saved data to {full_filepath}")
            print(f"  X shape: {X.shape}")
            print(f"  Y (predicted) shape: {Y.shape}")
            print(f"  Y_true (actual) shape: {Y_true_train_data.shape}")
            print(f"Saving took {end_time - start_time:.2f} seconds.")

            file_size_mb = os.path.getsize(full_filepath) / (1024**2)
            print(f"File size: {file_size_mb:.2f} MB")

        except Exception as e:
            print(f"An error occurred while saving the file: {e}")
else:
    missing_vars = []
    if "X" not in locals() or X is None:
        missing_vars.append("'X'")
    if "Y" not in locals() or Y is None:
        missing_vars.append("'Y' (predictions)")
    if "Y_true_train_data" not in locals() or Y_true_train_data is None:
        missing_vars.append("'Y_true_train_data' (true labels)")
    print(
        f"Error: Variables {', '.join(missing_vars)} not found or are None. Cannot save."
    )

# メモリを解放したい場合は、ここで X や Y, Y_true_train_data を削除することも検討
# del X
# del Y
# del Y_true_train_data
# print("Variables X, Y, and Y_true_train_data deleted from memory.")


## 画像データX, 予測値Y, 正解値の読み込み

In [ ]:
import os

import numpy as np

# 保存したファイル名を指定
filename = "gray_training_data_preprocessed_predicted_and_true_labels.npz"
load_directory = "../20250425_data"  # 保存したディレクトリと同じ場所を指定
full_filepath = os.path.join(load_directory, filename)

# 読み込み時のイメージ
loaded_data = np.load(full_filepath)
X_loaded = loaded_data["x_data"]
Y_predicted_loaded = loaded_data["y_predicted_data"]
Y_true_loaded = loaded_data["y_true_actual_data"]


In [ ]:
print(X_loaded.shape, Y_predicted_loaded.shape, Y_true_loaded.shape)


In [ ]:
# print("\nComparing elements using np.array_equal()...")
# are_equal = np.array_equal(X, X_loaded)

# if are_equal:
#     print("\nResult: SUCCESS! X and X_loaded have the same shape and elements.")
# else:
#     print(
#         "\nResult: FAILURE! X and X_loaded do NOT have the same elements (even though shapes match)."
#     )
#
# Comparing elements using np.array_equal()...

# Result: SUCCESS! X and X_loaded have the same shape and elements.


## AIMEの実装

In [ ]:
import cupy as cp
from sklearn.preprocessing import StandardScaler

from aime_xai.core import AIME


In [ ]:
aime = AIME()


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_loaded)
Y_predicted_scaled = scaler.fit_transform(Y_predicted_loaded)


In [ ]:
print(f"Shape of X_scaled: {X_scaled.shape}")
print(f"Data type of X_scaled: {X_scaled.dtype}")
print(f"Estimated memory size of X_scaled (CPU): {X_scaled.nbytes / (1024**3):.2f} GB")


In [ ]:
# X_cp = cp.asarray(X_scaled)
# Y_cp = cp.asarray(Y_predicted_loaded)
# if Y_cp.ndim == 1:
#     print(f"Reshaping Y_cp to ({Y_cp.shape[0]}, 1)")
#     Y_cp = Y_cp.reshape(-1, 1)


In [ ]:
# print(X_cp.shape, Y_cp.shape)


In [ ]:
aime.create_explainer(X_scaled, Y_predicted_scaled, normalize=False)


In [ ]:
aime.A_dagger.shape


In [ ]:
feature_names = [f"pixel_{i}" for i in range(X_scaled.shape[1])]


In [ ]:
import matplotlib.pyplot as plt


## 大局的特徴重要度

In [ ]:
# グローバル特徴重要度の計算（可視化なし）
df_global_importance = aime.global_feature_importance_without_viz(
    feature_names=feature_names,
    class_names=["Torque"],  # 回帰問題なので1つの出力
    top_k=None,  # 全ての特徴を使用
    top_k_criterion="average",
)

print(f"グローバル特徴重要度の形状: {df_global_importance.shape}")
print(f"出力次元: {df_global_importance.index.tolist()}")

# show_global_signed_gridを使用して可視化
# グレースケール画像（1チャンネル）で、回帰問題（1つの出力）なので1x1グリッド
aime.show_global_signed_grid(
    df_global=df_global_importance,
    class_names=["Torque"],  # 回帰問題の出力名
    H=IMG_HEIGHT,  # 224
    W=IMG_WIDTH,  # 224
    n_channels=1,  # グレースケール画像
    rows=1,  # 出力が1つなので1行
    cols=1,  # 出力が1つなので1列
    fname=None,  # 保存する場合はファイル名を指定（例: "global_importance.png"）
    mode="maxabs",  # 集約方法: "maxabs" または "l2"
)

print("グローバル特徴重要度の可視化が完了しました。")
print(
    "このヒートマップは、モデルがトルク予測のために画像のどの領域を重視しているかを示しています。"
)


## 局所的特徴重要度

In [ ]:
# 局所的特徴重要度の可視化例
# 分析したいサンプルのインデックスを指定
index_to_analyze = 10  # ここで分析したい画像のインデックスを指定

# 元画像データの取得（前処理済み）
x_sample_flat = X_loaded[index_to_analyze]  # フラット化された画像データ
y_predicted = Y_predicted_loaded[index_to_analyze]  # 予測値
y_true = Y_true_loaded[index_to_analyze]  # 正解値

# 元画像をリシェイプ（表示用）
# グレースケール画像なので (H, W) に変換
x_sample_2d = x_sample_flat.reshape(IMG_HEIGHT, IMG_WIDTH)

# ローカル特徴重要度の計算
# AIMEのlocal_feature_importance_without_vizを使用
local_importance_df = aime.local_feature_importance_without_viz(
    x=X_scaled[index_to_analyze],  # スケール済みデータを使用
    y=np.array([y_predicted]),  # 予測値（1次元配列として渡す）
    feature_names=feature_names,
    scale=False,  # 既にスケール済みなのでFalse
    scaler=None,
    ignore_zero_features=True,
)

# DataFrameから重要度の値を取得してリシェイプ
local_importance_flat = local_importance_df.iloc[0].values
local_importance_2d = local_importance_flat.reshape(IMG_HEIGHT, IMG_WIDTH)

# show_local_with_originalを使用して可視化
aime.show_local_with_original(
    orig_img3=x_sample_2d,  # 元画像（2D）
    contrib_img3=local_importance_2d,  # 局所特徴重要度（2D）
    true_name=f"True: {y_true:.4f}, Pred: {y_predicted:.4f}",  # ラベル情報
    fname=None,  # 保存する場合はファイル名を指定
    mode="maxabs",  # 集約方法: "maxabs" または "l2"
)

print(f"サンプル {index_to_analyze} の可視化が完了しました。")
print(f"正解値: {y_true:.4f}, 予測値: {y_predicted:.4f}")


NameError: name 'aime' is not defined

In [ ]:
# index_to_analyze = 900  # ここで分析したい画像のインデックスを指定
# x_sample = X_loaded[index_to_analyze]
# y_sample = Y_predicted_loaded[index_to_analyze]
# y_true_sample = Y_true_loaded[index_to_analyze]
# x_sample_cp = cp.asarray(x_sample)
# y_sample_cp = cp.array([y_sample])

# # ローカル特徴重要度の計算
# df_local_importance = aime_gpu.local_feature_importance_without_viz(
#     x_sample_cp,
#     y_sample_cp,
#     feature_names=feature_names,
#     scale=True,
#     scaler=scaler,
#     ignore_zero_features=True,
# )

# import cupy as cp  # df_local_importance が CuPy 配列の場合に備えて
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd  # df_local_importance が DataFrame の場合

# # ----- STEP 0: 必要な変数の定義 (実際の値に置き換えてください) -----
# # これらは通常、ノートブックの前のセルで定義・計算・ロードされているはずです。

# # 例: 画像の寸法 (VGG16の場合など)
# IMG_HEIGHT = 224
# IMG_WIDTH = 224
# CHANNELS = 3  # RGB画像の場合。グレースケールなら1

# # ----- STEP 1: 分析対象のデータ準備 -----

# # --- 画像データの取得とリシェイプ ---
# if not all(
#     var in locals() or var in globals()
#     for var in ["IMG_HEIGHT", "IMG_WIDTH", "CHANNELS"]
# ):
#     print("Error: IMG_HEIGHT, IMG_WIDTH, or CHANNELS not defined.")
#     raise NameError(
#         "Image dimensions (IMG_HEIGHT, IMG_WIDTH, CHANNELS) are not defined."
#     )

# if (
#     "X_loaded" not in locals()
#     or X_loaded is None
#     or not isinstance(X_loaded, np.ndarray)
# ):
#     print("Error: 'X_loaded' not found, is None, or is not a NumPy array.")
#     raise NameError("'X_loaded' is not properly defined or loaded.")

# if not (0 <= index_to_analyze < X_loaded.shape[0]):
#     print(
#         f"Error: index_to_analyze ({index_to_analyze}) is out of range for X_loaded (0 to {X_loaded.shape[0]-1})."
#     )
#     raise IndexError("index_to_analyze is out of bounds.")

# x_sample_flat_preprocessed = X_loaded[index_to_analyze]
# try:
#     target_size = IMG_HEIGHT * IMG_WIDTH * CHANNELS
#     if (
#         CHANNELS == 1 and x_sample_flat_preprocessed.size == IMG_HEIGHT * IMG_WIDTH
#     ):  # グレースケールで平坦化済みの場合
#         target_size = IMG_HEIGHT * IMG_WIDTH
#         x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
#             IMG_HEIGHT, IMG_WIDTH, 1
#         )  # チャンネル次元を保持
#     elif x_sample_flat_preprocessed.size == target_size:
#         x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
#             IMG_HEIGHT, IMG_WIDTH, CHANNELS
#         )
#     else:
#         raise ValueError(
#             f"Flat image size {x_sample_flat_preprocessed.size} does not match dimensions "
#             f"({IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS} or {IMG_HEIGHT}x{IMG_WIDTH} for grayscale)."
#         )
#     print(
#         f"Original preprocessed x_sample (index {index_to_analyze}) reshaped to: {x_sample_reshaped_preprocessed.shape}"
#     )
# except ValueError as ve:
#     print(f"Error reshaping x_sample: {ve}")
#     raise

# # --- 特徴重要度の取得とリシェイプ ---
# if "df_local_importance" not in locals() or df_local_importance is None:
#     print("Error: 'df_local_importance' not found or is None.")
#     print(
#         "Please ensure local feature importance has been calculated for the selected sample."
#     )
#     raise NameError("'df_local_importance' is not defined.")

# if isinstance(df_local_importance, pd.DataFrame):
#     if (
#         df_local_importance.shape[0] == 1 and df_local_importance.shape[1] > 0
#     ):  # 単一サンプルのDataFrameを想定
#         local_importance_values_flat = df_local_importance.iloc[0].to_numpy()
#     elif (
#         df_local_importance.shape[0] > 1
#         and df_local_importance.shape[0] > index_to_analyze
#     ):  # 複数行ある場合
#         print(
#             f"Warning: df_local_importance has multiple rows. Selecting row at index {index_to_analyze}."
#         )
#         local_importance_values_flat = df_local_importance.iloc[
#             index_to_analyze
#         ].to_numpy()
#     else:
#         raise ValueError(
#             f"df_local_importance shape {df_local_importance.shape} is not suitable or index is out of bounds."
#         )
# elif isinstance(df_local_importance, cp.ndarray):
#     # CuPy配列の場合、1サンプル分になっているか確認が必要
#     if df_local_importance.ndim == 1:
#         local_importance_values_flat = cp.asnumpy(df_local_importance)
#     elif df_local_importance.ndim == 2 and df_local_importance.shape[0] == 1:
#         local_importance_values_flat = cp.asnumpy(df_local_importance.flatten())
#     elif (
#         df_local_importance.ndim == 2
#         and df_local_importance.shape[0] > index_to_analyze
#     ):
#         print(
#             f"Warning: CuPy df_local_importance has multiple rows. Selecting row at index {index_to_analyze}."
#         )
#         local_importance_values_flat = cp.asnumpy(
#             df_local_importance[index_to_analyze].flatten()
#         )
#     else:
#         raise ValueError(
#             f"CuPy df_local_importance shape {df_local_importance.shape} is not suitable."
#         )
# elif isinstance(df_local_importance, np.ndarray):
#     if df_local_importance.ndim == 1:
#         local_importance_values_flat = df_local_importance
#     elif df_local_importance.ndim == 2 and df_local_importance.shape[0] == 1:
#         local_importance_values_flat = df_local_importance.flatten()
#     elif (
#         df_local_importance.ndim == 2
#         and df_local_importance.shape[0] > index_to_analyze
#     ):
#         print(
#             f"Warning: NumPy df_local_importance has multiple rows. Selecting row at index {index_to_analyze}."
#         )
#         local_importance_values_flat = df_local_importance[index_to_analyze].flatten()
#     else:
#         raise ValueError(
#             f"NumPy df_local_importance shape {df_local_importance.shape} is not suitable."
#         )
# else:
#     raise TypeError("df_local_importance is not of a recognized type.")

# try:
#     # 特徴重要度がピクセル単位 (H*W) か、チャネル含む (H*W*C) かで処理を分ける
#     if local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH:
#         importance_heatmap_2d = local_importance_values_flat.reshape(
#             IMG_HEIGHT, IMG_WIDTH
#         )
#     elif local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH * CHANNELS:
#         temp_heatmap_3d = local_importance_values_flat.reshape(
#             IMG_HEIGHT, IMG_WIDTH, CHANNELS
#         )
#         importance_heatmap_2d = np.mean(temp_heatmap_3d, axis=2)  # チャネル平均
#         print("Importance values reshaped from (H,W,C) by taking mean over channels.")
#     else:
#         raise ValueError(
#             f"Size of local_importance_values_flat ({local_importance_values_flat.size}) "
#             f"does not match expected image dimensions (H*W or H*W*C)."
#         )
#     print(f"Reshaped importance heatmap to: {importance_heatmap_2d.shape}")
# except ValueError as ve:
#     print(f"Error reshaping importance_heatmap_2d: {ve}")
#     raise

# # ----- STEP 1.5: 特徴重要度の正規化 -----
# all_importance_values_for_norm = (
#     importance_heatmap_2d.flatten()
# )  # 2Dヒートマップから正規化範囲を計算

# local_min = all_importance_values_for_norm.min()
# local_max = all_importance_values_for_norm.max()
# print(
#     f"Calculated local_min: {local_min:.4f}, local_max: {local_max:.4f} from the reshaped importance heatmap."
# )

# if (local_max - local_min) > 1e-6:
#     importance_heatmap_normalized = (importance_heatmap_2d - local_min) / (
#         local_max - local_min
#     )
# else:
#     importance_heatmap_normalized = np.zeros_like(importance_heatmap_2d)
#     print(
#         "Warning: All importance values in the heatmap are nearly identical. Normalized heatmap will be all zeros."
#     )
# importance_heatmap_normalized = np.clip(importance_heatmap_normalized, 0, 1)
# print(f"Shape of normalized importance heatmap: {importance_heatmap_normalized.shape}")


# # ----- STEP 2: 表示用関数の定義 -----
# def display_denormalized_vgg16_for_subplot(ax, img_data_reshaped_preprocessed, title):
#     if not (
#         img_data_reshaped_preprocessed.ndim == 3
#         and img_data_reshaped_preprocessed.shape[-1] == 3
#     ):
#         ax.text(
#             0.5,
#             0.5,
#             f"Cannot display original image:\nExpected 3D RGB image,\ngot shape {img_data_reshaped_preprocessed.shape}",
#             ha="center",
#             va="center",
#             fontsize=9,
#         )
#         ax.set_title(title, fontsize=10)
#         ax.axis("off")
#         return

#     img_restored = img_data_reshaped_preprocessed.copy()
#     # VGG16の逆前処理: 平均値を足し、BGRをRGBに変換
#     img_restored[..., 0] += 103.939  # Blue
#     img_restored[..., 1] += 116.779  # Green
#     img_restored[..., 2] += 123.68  # Red
#     img_restored = img_restored[..., ::-1]  # BGR to RGB
#     img_restored = np.clip(img_restored, 0, 255).astype(
#         np.uint8
#     )  # 0-255にクリップし、uint8に変換
#     ax.imshow(img_restored)
#     ax.set_title(title, fontsize=10)
#     ax.axis("off")


# # ----- STEP 3: 図の表示 -----
# fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))  # figsizeを少し調整

# # --- サブプロット1: 元画像の表示 ---
# title_original = f"Original Image (Index: {index_to_analyze})"
# if x_sample_reshaped_preprocessed.shape[-1] == 3:  # RGB
#     display_denormalized_vgg16_for_subplot(
#         axes[0],
#         x_sample_reshaped_preprocessed,
#         title_original + "\n(VGG16 De-normalized)",
#     )
# elif x_sample_reshaped_preprocessed.shape[-1] == 1:  # Grayscale
#     img_min_orig = x_sample_reshaped_preprocessed.min()
#     img_max_orig = x_sample_reshaped_preprocessed.max()
#     epsilon_orig = 1e-6
#     if (img_max_orig - img_min_orig) > epsilon_orig:
#         scaled_image_orig = (x_sample_reshaped_preprocessed - img_min_orig) / (
#             img_max_orig - img_min_orig + epsilon_orig
#         )
#     else:
#         scaled_image_orig = np.zeros_like(x_sample_reshaped_preprocessed)
#     scaled_image_orig = np.clip(scaled_image_orig, 0, 1)
#     axes[0].imshow(scaled_image_orig.squeeze(), cmap="gray")
#     axes[0].set_title(title_original + "\n(Grayscale Scaled)", fontsize=10)
#     axes[0].axis("off")
# else:
#     axes[0].text(
#         0.5,
#         0.5,
#         "Cannot display original image\n(Unsupported channel num)",
#         ha="center",
#         va="center",
#         fontsize=9,
#     )
#     axes[0].set_title(title_original, fontsize=10)
#     axes[0].axis("off")

# # --- サブプロット2: 正規化されたヒートマップのみを表示 ---
# im = axes[1].imshow(
#     importance_heatmap_normalized,
#     cmap="inferno",
#     interpolation="bilinear",
#     vmin=0,
#     vmax=1,
# )
# axes[1].set_title(
#     f"Normalized Importance Heatmap\n(Index: {index_to_analyze})", fontsize=10
# )
# axes[1].axis("off")

# # カラーバーを追加
# fig.colorbar(
#     im,
#     ax=axes[1],
#     orientation="vertical",
#     fraction=0.046,
#     pad=0.04,
#     label="Normalized Importance (0-1)",
# )

# plt.suptitle(
#     f"Local Feature Importance Analysis for Sample {index_to_analyze}", fontsize=14
# )  # 全体のタイトル
# plt.tight_layout(rect=[0, 0, 1, 0.96])  # rectでsuptitleとの重なりを調整
# plt.show()


## 100個ランダムにLocalを保存

In [ ]:
import os

import cupy as cp  # type: ignore
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# ----- STEP 0: 必要な変数の定義と設定 -----

# ★★★ CHANNELS を 1 に変更 ★★★
IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 1  # グレースケール画像

output_directory = "/home/kosukeyano/TUT_DTC Dropbox/Yano Kosuke/AIME/EM/Local_Feature_Importance_gray_1channel"  # ディレクトリ名を適宜変更
os.makedirs(output_directory, exist_ok=True)

# ----- AIME_GPU オブジェクトと関連変数の準備 -----
# この部分は、ご自身の環境で AIME_GPU オブジェクト (`aime_gpu`)、
# 特徴名リスト (`feature_names`)、スケーラー (`scaler`、任意) を
# 適切に初期化・準備しているものとします。
# X_loaded, Y_predicted_loaded, Y_true_loaded も .npz ファイルなどから
# 既にロードされている必要があります。

# ダミーデータの例 (実際のデータで置き換えてください)
# もしこれらの変数が未定義の場合、以下のダミーデータでテストできます。
# 実行前に、実際のデータロード処理があることを確認してください。
# num_samples_dummy = 110  # end_index よりも大きい値
# if "X_loaded" not in locals():
#     print("警告: 'X_loaded' が未定義です。ダミーデータを作成します。")
#     if CHANNELS == 1:
#         X_loaded = np.random.rand(num_samples_dummy, IMG_HEIGHT * IMG_WIDTH).astype(
#             np.float32
#         )
#     else:  # CHANNELS == 3
#         X_loaded = np.random.rand(
#             num_samples_dummy, IMG_HEIGHT * IMG_WIDTH * CHANNELS
#         ).astype(np.float32)
#     print(f"ダミー X_loaded shape: {X_loaded.shape}")

# if "Y_true_loaded" not in locals():
#     print("警告: 'Y_true_loaded' が未定義です。ダミーデータを作成します。")
#     Y_true_loaded = np.random.rand(num_samples_dummy).astype(np.float32)
#     print(f"ダミー Y_true_loaded shape: {Y_true_loaded.shape}")

# if "Y_predicted_loaded" not in locals():
#     print("警告: 'Y_predicted_loaded' が未定義です。ダミーデータを作成します。")
#     Y_predicted_loaded = Y_true_loaded * (
#         1 + 0.1 * (np.random.rand(num_samples_dummy) - 0.5)
#     )  # 真の値に近い予測値
#     print(f"ダミー Y_predicted_loaded shape: {Y_predicted_loaded.shape}")

# # AIME関連の変数のダミー (実際のオブジェクトで置き換えてください)
# if "aime_gpu" not in locals():
#     print("警告: 'aime_gpu' が未定義です。ダミーオブジェクトを作成します。")

#     class DummyAIMEGPU:  # AIMEのAPIを模倣したダミークラス
#         def local_feature_importance_without_viz(
#             self, x_sample, y_sample, feature_names, scale, scaler, ignore_zero_features
#         ):
#             print("DummyAIMEGPU: local_feature_importance_without_viz called.")
#             # 特徴量の数に基づいてランダムな重要度を返す
#             num_features = x_sample.shape[0]  # x_sampleはフラット化されていると仮定
#             importance = cp.random.rand(num_features).astype(cp.float32)
#             return importance  # CuPy配列を返す

#     aime_gpu = DummyAIMEGPU()

# if "feature_names" not in locals():
#     print("警告: 'feature_names' が未定義です。ダミーの特徴名リストを作成します。")
#     if CHANNELS == 1:
#         num_total_pixels = IMG_HEIGHT * IMG_WIDTH
#     else:  # CHANNELS == 3 (元のロジック)
#         num_total_pixels = IMG_HEIGHT * IMG_WIDTH * CHANNELS
#     feature_names = [f"pixel_{i}" for i in range(num_total_pixels)]
#     print(f"ダミー feature_names length: {len(feature_names)}")

# if "scaler" not in locals():
#     scaler = None  # スケーラーはオプションなので None でも可


# ----- ループ処理 -----
start_index = 1
end_index = 100

print(
    f"\nStarting to generate and save images for indices {start_index} to {end_index}...\n"
)

for index_to_analyze in range(start_index, end_index + 1):
    print(f"--- Processing Sample Index: {index_to_analyze} ---")

    if not (0 <= index_to_analyze < X_loaded.shape[0]):
        print(
            f"Skipping: Index {index_to_analyze} is out of range for X_loaded (max index: {X_loaded.shape[0] - 1})."
        )
        continue

    try:
        # ----- STEP 1 (ループ内): 分析対象のデータ準備 -----
        x_sample_flat_preprocessed = X_loaded[
            index_to_analyze
        ]  # (IMG_HEIGHT * IMG_WIDTH * CHANNELS_LOADED,)
        actual_value = Y_true_loaded[index_to_analyze]
        predicted_value = Y_predicted_loaded[index_to_analyze]

        # 入力データがフラットであることを確認
        expected_flat_size = (
            IMG_HEIGHT * IMG_WIDTH * CHANNELS
        )  # 現在のCHANNELS設定 (1) で期待されるサイズ
        if (
            CHANNELS == 1 and x_sample_flat_preprocessed.size == IMG_HEIGHT * IMG_WIDTH
        ):  # 元々フラットなグレースケールだった場合
            pass  # そのまま使用
        elif x_sample_flat_preprocessed.size != expected_flat_size:
            # もしX_loadedが(N, H, W, C_orig)や(N, H*W*C_orig)でロードされていて、
            # 現在のCHANNELS=1と異なる場合の対応が必要になるかもしれない。
            # ここでは、X_loadedが既に (N, H*W*1) または (N, H*W) の形式であることを期待する。
            print(
                f"Warning: x_sample_flat_preprocessed size {x_sample_flat_preprocessed.size} "
                f"does not match expected flat size for CHANNELS={CHANNELS} ({expected_flat_size}). "
                f"Assuming it's already correct for CHANNELS=1."
            )
            if (
                x_sample_flat_preprocessed.size != IMG_HEIGHT * IMG_WIDTH
            ):  # 厳密にH*Wであることを確認
                raise ValueError(
                    f"Flat image size {x_sample_flat_preprocessed.size} does not match H*W ({IMG_HEIGHT * IMG_WIDTH})."
                )

        x_sample_cp = cp.asarray(x_sample_flat_preprocessed)
        y_for_aime_cp = cp.array([predicted_value])

        current_df_local_importance = aime_gpu.local_feature_importance_without_viz(
            x_sample_cp,
            y_for_aime_cp,
            feature_names=feature_names,  # CHANNELS=1 用の長さになっている必要がある
            scale=True,
            scaler=scaler,
            ignore_zero_features=True,
        )
        print(
            f"Local feature importance calculated. Actual: {actual_value:.4f}, Predicted: {predicted_value:.4f}"
        )

        # 画像データのリシェイプ
        # x_sample_flat_preprocessed は (H*W,) または (H*W*1,) のフラットな配列と仮定
        try:
            x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
                IMG_HEIGHT, IMG_WIDTH, CHANNELS
            )
        except ValueError as e:
            # もしx_sample_flat_preprocessedが(H*W)で、CHANNELS=1の場合、(H,W,1)にしたい
            if (
                x_sample_flat_preprocessed.size == IMG_HEIGHT * IMG_WIDTH
                and CHANNELS == 1
            ):
                x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
                    IMG_HEIGHT, IMG_WIDTH, 1
                )
            else:
                raise ValueError(
                    f"Cannot reshape flat image of size {x_sample_flat_preprocessed.size} to "
                    f"({IMG_HEIGHT}, {IMG_WIDTH}, {CHANNELS}). Original error: {e}"
                )
        print(f"Reshaped original image to: {x_sample_reshaped_preprocessed.shape}")

        # 特徴重要度の取得とリシェイプ
        if isinstance(
            current_df_local_importance, pd.DataFrame
        ):  # AIMEがDataFrameを返す場合
            local_importance_values_flat = current_df_local_importance.iloc[
                0
            ].to_numpy()
        elif isinstance(
            current_df_local_importance, cp.ndarray
        ):  # AIMEがCuPy配列を返す場合
            local_importance_values_flat = cp.asnumpy(
                current_df_local_importance.flatten()
            )
        elif isinstance(
            current_df_local_importance, np.ndarray
        ):  # AIMEがNumPy配列を返す場合
            local_importance_values_flat = current_df_local_importance.flatten()
        else:
            raise TypeError(
                f"current_df_local_importance is of unexpected type: {type(current_df_local_importance)}"
            )

        # CHANNELS=1 の場合、特徴重要度マップは (IMG_HEIGHT * IMG_WIDTH) のサイズのはず
        if local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH:
            importance_heatmap_2d = local_importance_values_flat.reshape(
                IMG_HEIGHT, IMG_WIDTH
            )
        # (元の3チャネル用ロジックは CHANNELS=1 では通常通らないが、念のため残す)
        elif (
            local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH * CHANNELS
            and CHANNELS == 3
        ):  # 元のCHANNELSが3だった場合
            temp_heatmap_3d = local_importance_values_flat.reshape(
                IMG_HEIGHT, IMG_WIDTH, CHANNELS
            )
            importance_heatmap_2d = np.mean(
                temp_heatmap_3d, axis=2
            )  # 3チャネルの平均を取る
        else:
            raise ValueError(
                f"Size of local_importance_values_flat ({local_importance_values_flat.size}) "
                f"does not match expected image dimensions for CHANNELS={CHANNELS} "
                f"(H*W = {IMG_HEIGHT * IMG_WIDTH} or H*W*C = {IMG_HEIGHT * IMG_WIDTH * CHANNELS})."
            )
        print(f"Reshaped importance heatmap to: {importance_heatmap_2d.shape}")

        # 特徴重要度の正規化 (変更なし)
        all_importance_values_for_norm = importance_heatmap_2d.flatten()
        local_min = all_importance_values_for_norm.min()
        local_max = all_importance_values_for_norm.max()
        if (local_max - local_min) > 1e-6:  # avoid division by zero or near-zero
            importance_heatmap_normalized = (importance_heatmap_2d - local_min) / (
                local_max - local_min
            )
        else:
            importance_heatmap_normalized = np.zeros_like(
                importance_heatmap_2d
            )  # If all values are same
        importance_heatmap_normalized = np.clip(importance_heatmap_normalized, 0, 1)

        # ----- STEP 3 (ループ内): 図の表示と保存 -----
        fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))  # 1行2列

        # サブプロット1: 元画像の表示
        title_original = f"Original Image (Index: {index_to_analyze})"
        if x_sample_reshaped_preprocessed.shape[-1] == 1:  # グレースケール画像
            # 元画像がどのような前処理をされているかによってスケーリング方法を調整
            # 例: もし0-1にスケーリング済みならそのまま表示
            # 例: もし-1から1なら (img + 1)/2 で0-1に
            # ここでは、単純にimshowできる形式を期待 (例: 0-1 または 0-255 のグレースケール)
            # X_loaded の前処理に合わせる必要がある
            # tf.keras.applications.vgg16.preprocess_input のような複雑な前処理はグレースケールでは通常しない
            # ここでは、元のピクセル値が [0, 255] または [0, 1] であると仮定して表示
            img_to_display_orig = (
                x_sample_reshaped_preprocessed.squeeze()
            )  # (H, W, 1) -> (H, W)

            # ピクセル値の範囲に基づいて表示を調整 (例)
            if img_to_display_orig.min() >= 0 and img_to_display_orig.max() <= 1:
                cmap_orig = "gray"
                vmin_orig, vmax_orig = 0, 1
            elif img_to_display_orig.min() >= 0 and img_to_display_orig.max() <= 255:
                cmap_orig = "gray"
                vmin_orig, vmax_orig = 0, 255
            else:  # それ以外の範囲なら、0-1に正規化して表示
                img_min_orig, img_max_orig = (
                    img_to_display_orig.min(),
                    img_to_display_orig.max(),
                )
                epsilon_orig = 1e-6
                if (img_max_orig - img_min_orig) > epsilon_orig:
                    img_to_display_orig = (img_to_display_orig - img_min_orig) / (
                        img_max_orig - img_min_orig + epsilon_orig
                    )
                else:
                    img_to_display_orig = np.zeros_like(img_to_display_orig)
                img_to_display_orig = np.clip(img_to_display_orig, 0, 1)
                cmap_orig = "gray"
                vmin_orig, vmax_orig = 0, 1

            axes[0].imshow(
                img_to_display_orig, cmap=cmap_orig, vmin=vmin_orig, vmax=vmax_orig
            )
            axes[0].set_title(title_original, fontsize=10)
            axes[0].axis("off")
        elif (
            x_sample_reshaped_preprocessed.shape[-1] == 3
        ):  # 3チャネル画像 (VGG16逆前処理用)
            # この部分はCHANNELS=1の時は通らないはずだが、コードとして残す
            display_denormalized_vgg16_for_subplot(
                axes[0], x_sample_reshaped_preprocessed, title_original
            )
        else:
            axes[0].text(
                0.5,
                0.5,
                f"Cannot display original image\nUnsupported shape: {x_sample_reshaped_preprocessed.shape}",
                ha="center",
                va="center",
                fontsize=9,
            )
            axes[0].set_title(title_original, fontsize=10)
            axes[0].axis("off")

        # サブプロット1の下に 正解値と予測値を表示 (変更なし)
        axes[0].text(
            0.5,
            -0.08,
            f"Actual Value (True): {actual_value:.4f}\nPredicted Value: {predicted_value:.4f}",
            size=10,
            ha="center",
            transform=axes[0].transAxes,
            bbox=dict(boxstyle="round,pad=0.3", fc="aliceblue", alpha=0.9),
        )

        # サブプロット2: 正規化されたヒートマップのみを表示 (変更なし)
        im = axes[1].imshow(
            importance_heatmap_normalized,
            cmap="inferno",
            interpolation="bilinear",
            vmin=0,
            vmax=1,
        )
        axes[1].set_title(
            f"Normalized Importance Heatmap\n(Index: {index_to_analyze})", fontsize=10
        )
        axes[1].axis("off")
        fig.colorbar(
            im,
            ax=axes[1],
            orientation="vertical",
            fraction=0.046,
            pad=0.04,
            label="Normalized Importance (0-1)",
        )

        fig.suptitle(
            f"Local Feature Importance - Sample {index_to_analyze}", fontsize=14
        )
        plt.tight_layout(
            rect=[0, 0.03, 1, 0.95]
        )  # タイトルとテキストが重ならないように調整

        save_filename = f"heatmap_sample_{index_to_analyze:03d}.png"
        full_save_path = os.path.join(output_directory, save_filename)
        plt.savefig(full_save_path, bbox_inches="tight")
        print(f"Saved: {full_save_path}")
        plt.close(fig)  # メモリ解放のために図を閉じる

    except Exception as e:
        print(f"Error processing sample index {index_to_analyze}: {e}")
        import traceback

        traceback.print_exc()  # 詳細なエラー情報を表示

print("\n--- All specified samples processed. ---")
